In [3]:
import pyodbc
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error
import optuna
import shap
from sklearn.model_selection import train_test_split, cross_val_score
from category_encoders.target_encoder import TargetEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. Connect to the database and fetch the data
conn = pyodbc.connect(
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)

query = """
SELECT 
[call_type],
[priority],
[initial_call_type_mapping],
cast([cad_event_original_time_queued_date] as date) as  cad_event_original_time_queued_date,
cast([cad_event_original_time_queued_datetime_hour] as float) as cad_event_original_time_queued_datetime_hour,
[dispatch_precinct],
[dispatch_sector],
[dispatch_beat],
[dispatch_reporting_area],
[cad_event_response_category],
[call_type_indicator],
[dispatch_neighborhood],
[call_type_received_classification],
[call_sign_total_service_time_s]
from [gt].[dbo].[call_data_20251019_processed_v44]
tablesample (10 percent)
"""

df = pd.read_sql(query, conn)
conn.close()

# 2. Preprocess the data
df['cad_event_original_time_queued_date'] = pd.to_datetime(df['cad_event_original_time_queued_date'])
df['day_of_week'] = df['cad_event_original_time_queued_date'].dt.dayofweek
df['month'] = df['cad_event_original_time_queued_date'].dt.month
df['day'] = df['cad_event_original_time_queued_date'].dt.day
df['year'] = df['cad_event_original_time_queued_date'].dt.year
df = df.drop('cad_event_original_time_queued_date', axis=1)

df['log_target'] = np.log1p(data['call_sign_total_service_time_s'])
df = df.drop('call_sign_total_service_time_s', axis=1)

# 3. Define features and target
X = df.drop('log_target', axis=1)
y = df['log_target']

# 4. Identify numerical and categorical columns
numerical_cols = [
    'cad_event_original_time_queued_datetime_hour',
    'day_of_week',
    'month',
    'day',
    'year'
]
categorical_cols = [col for col in X.columns if col not in numerical_cols]

# 5. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 6. Preprocess with ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# 7. Train XGBoost model
model = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train_preprocessed, y_train)

# 8. Evaluate the model
y_pred = model.predict(X_test_preprocessed)
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse:.2f}")

C:\Users\RQ\AppData\Local\Temp\ipykernel_23792\1084283869.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


Mean Squared Error: 5184275.07


In [18]:
import pyodbc
import pandas as pd
import xgboost as xgb
import optuna
import shap
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from category_encoders.target_encoder import TargetEncoder

# 1. Connect to the database and fetch the data
conn = pyodbc.connect(
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)

query = """
SELECT 
[call_type],
[priority],
[initial_call_type],
cast([cad_event_original_time_queued_date] as date) as  cad_event_original_time_queued_date,
cast([cad_event_original_time_queued_datetime_hour] as float) as cad_event_original_time_queued_datetime_hour,
[dispatch_precinct],
[dispatch_sector],
[dispatch_beat],
[dispatch_reporting_area],
[cad_event_response_category],
[call_type_indicator],
[dispatch_neighborhood],
[call_type_received_classification],
[call_sign_total_service_time_s]
from [gt].[dbo].[call_data_20251019_processed_v44]
tablesample (10 percent)
"""

df = pd.read_sql(query, conn)
conn.close()

df = df.drop(columns=["cad_event_response_category"])

categorical_cols = [
    "call_type", "initial_call_type", "priority", "call_type_indicator", "dispatch_precinct",
    "dispatch_sector", "dispatch_beat", "call_type_indicator", "dispatch_reporting_area", "dispatch_neighborhood",
    "call_type_received_classification"
]
for col in categorical_cols:
    df[col] = df[col].fillna("UNKNOWN")

#2. Feature Engineering
df["cad_event_original_time_queued_date"] = pd.to_datetime(df["cad_event_original_time_queued_date"])
df["day_of_week"] = df["cad_event_original_time_queued_date"].dt.dayofweek  # 0=Monday, 6=Sunday
df["month"] = df["cad_event_original_time_queued_date"].dt.month
df["hour"] = df["cad_event_original_time_queued_datetime_hour"].astype(int)

# Drop original datetime columns
df = df.drop(columns=["cad_event_original_time_queued_date", "cad_event_original_time_queued_datetime_hour"])

#3. Target Variable
# Log transform target to stabilize variance
df["log_call_sign_total_service_time_s"] = np.log1p(df["call_sign_total_service_time_s"])

# 4. Train-Test Split
X = df.drop(columns=["call_sign_total_service_time_s", "log_call_sign_total_service_time_s"])
y = df["log_call_sign_total_service_time_s"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#5. Column Transformer for Categorical/Continuous Features
# Define categorical and numerical features
categorical_features = [
    "call_type", "priority", "call_type_indicator", "dispatch_precinct",
    "dispatch_sector", "dispatch_reporting_area", "dispatch_neighborhood",
    "call_type_received_classification"
]
numerical_features = ["day_of_week", "month", "hour"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", TargetEncoder(), categorical_features),
        ("num", "passthrough", numerical_features)
    ])

# Fit and transform data
X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

# 6. Hyperparameter Tuning with Optuna
def objective(trial):
    params = {
        "n_estimators": 500,
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 10),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10),
        "random_state": 42
    }
    model = xgb.XGBRegressor(**params)
    scores = cross_val_score(model, X_train_processed, y_train, cv=3, scoring="neg_mean_squared_error")
    return -scores.mean()

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)  # Increase trials for better tuning

best_params = study.best_params
print("Best Hyperparameters:", best_params)

#7. Train Final Model
final_model.fit(
    X_train_processed, y_train,
    eval_set=[(X_test_processed, y_test)],
    verbose=False
)

#8. Evaluate Model
y_pred = final_model.predict(X_test_processed)
y_true = df.loc[X_test.index, "call_sign_total_service_time_s"]
y_pred_exp = np.expm1(y_pred)  # Reverse log transformation

mse = mean_squared_error(y_true, y_pred_exp)
mae = mean_absolute_error(y_true, y_pred_exp)
r2 = r2_score(y_true, y_pred_exp)

print(f"MSE: {mse:.4f}, MAE: {mae:.4f}, R²: {r2:.4f}")

# --- 9. SHAP Analysis ---
#booster = final_model.get_booster()
#explainer = shap.Explainer(booster, X_train_processed)
#shap_values = explainer(X_test_processed)
#shap.summary_plot(shap_values, X_test_processed, feature_names=X.columns.tolist())

C:\Users\RQ\AppData\Local\Temp\ipykernel_23792\1120330642.py:43: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
[I 2025-11-06 17:19:01,748] A new study created in memory with name: no-name-d3f8fdbb-68d6-464d-9ba7-efdae5be2ec7
[I 2025-11-06 17:19:03,819] Trial 0 finished with value: 3.21580075313329 and parameters: {'max_depth': 9, 'learning_rate': 0.21645611359805972, 'subsample': 0.7323374293361227, 'colsample_bytree': 0.6791496924032023, 'reg_alpha': 1.2546871698513917, 'reg_lambda': 0.32467566629283406}. Best is trial 0 with value: 3.21580075313329.
[I 2025-11-06 17:19:04,955] Trial 1 finished with value: 2.7006190653351214 and parameters: {'max_depth': 7, 'learning_rate': 0.1438680971751941, 'subsample': 0.6808570416502882, 'colsample_bytree': 0.7066293228995484, 'reg_alpha': 3.8412146104212863, 're

Best Hyperparameters: {'max_depth': 7, 'learning_rate': 0.02055328005589143, 'subsample': 0.8970198395706704, 'colsample_bytree': 0.7762638829486583, 'reg_alpha': 5.027577371836275, 'reg_lambda': 3.2964679711804408}
MSE: 6976235.5256, MAE: 1742.3968, R²: -0.2330


In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# Step 1: Load your dataset (assume it's in a variable called df)
# df = pd.read_csv('your_dataset.csv')  # Uncomment and adjust as needed
conn = pyodbc.connect(
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)

query = """
SELECT 
[call_type],
[priority],
[initial_call_type],
cast([cad_event_original_time_queued_date] as date) as  cad_event_original_time_queued_date,
cast([cad_event_original_time_queued_datetime_hour] as float) as cad_event_original_time_queued_datetime_hour,
[dispatch_precinct],
[dispatch_sector],
[dispatch_beat],
[dispatch_reporting_area],
[cad_event_response_category],
[call_type_indicator],
[dispatch_neighborhood],
[call_type_received_classification],
case when [call_sign_total_service_time_s] between 0 and 600
then '0 - 10 min'
when [call_sign_total_service_time_s] between 600 and 1200
then '11 - 20 min'
when [call_sign_total_service_time_s] between 1200 and 1800
then '21 - 30 min'
when [call_sign_total_service_time_s] between 1800 and 2400
then '31 - 40 min'
when [call_sign_total_service_time_s] between 2400 and 3000
then '41 - 50 min'
when [call_sign_total_service_time_s] between 3000 and 3600
then '51 - 60 min'
when [call_sign_total_service_time_s] >= 3600
then '60+ min'
else NULL end as service_time_flag
from [gt].[dbo].[call_data_20251019_processed_v44]
tablesample (10 percent)
"""

df = pd.read_sql(query, conn)
conn.close()

df = df.drop(columns=["cad_event_response_category"])
# Step 2: Convert priority to integer
df['priority'] = df['priority'].astype(int)

# Step 3: Map the target variable to ordinal values
service_time_map = {
    '0 - 10 min': 0,
    '11 - 20 min': 1,
    '21 - 30 min': 2,
    '31 - 40 min': 3,
    '41 - 50 min': 4,
    '51 - 60 min': 5,
    '60+ min': 6
}
df['service_time_flag'] = df['service_time_flag'].map(service_time_map)

# Step 4: Split into features and target
X = df.drop('service_time_flag', axis=1)
y = df['service_time_flag']

# Step 5: Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 6: Encode categorical features
categorical_cols = [col for col in X_train.columns if X_train[col].dtype == 'object']

label_encoders = {}  # To store encoders for later use or inverse transforms

for col in categorical_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    label_encoders[col] = le  # Save encoder for potential future use

# Step 7: Train XGBoost classifier
model = XGBClassifier(
    use_label_encoder=False,
    eval_metric='mlogloss'
)
model.fit(X_train, y_train)

# Step 8: Evaluate the model
y_pred = model.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))

C:\Users\RQ\AppData\Local\Temp\ipykernel_23792\201553099.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


ValueError: y contains previously unseen labels: 'ASSIGNED DUTY - SEATTLE CENTER EVENT'

In [32]:
import pandas as pd
import pyodbc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# Load data
conn = pyodbc.connect(
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=QUAN;'
    r'DATABASE=gt;'
    r'Trusted_Connection=yes;'
)
query = """
SELECT
[call_type],
[priority],
[initial_call_type],
cast([cad_event_original_time_queued_date] as date) as  cad_event_original_time_queued_date,
cast([cad_event_original_time_queued_datetime_hour] as float) as cad_event_original_time_queued_datetime_hour,
[dispatch_precinct],
[dispatch_sector],
[dispatch_beat],
[dispatch_reporting_area],
[cad_event_response_category],
[call_type_indicator],
[dispatch_neighborhood],
[call_type_received_classification],
case when [call_sign_total_service_time_s] between 0 and 600
then '0 - 10 min'
when [call_sign_total_service_time_s] between 600 and 1200
then '11 - 20 min'
when [call_sign_total_service_time_s] between 1200 and 1800
then '21 - 30 min'
when [call_sign_total_service_time_s] between 1800 and 2400
then '31 - 40 min'
when [call_sign_total_service_time_s] between 2400 and 3000
then '41 - 50 min'
when [call_sign_total_service_time_s] between 3000 and 3600
then '51 - 60 min'
when [call_sign_total_service_time_s] >= 3600
then '60+ min'
else NULL end as service_time_flag
from [gt].[dbo].[call_data_20251019_processed_v44]
tablesample (10 percent)
"""
df = pd.read_sql(query, conn)
conn.close()

# Clean data
df = df.dropna(subset=['service_time_flag'])

# Encode target
le = LabelEncoder()
df['service_time_flag_encoded'] = le.fit_transform(df['service_time_flag'])

# Split data
X = df.drop(columns=['service_time_flag', 'service_time_flag_encoded'])
y = df['service_time_flag_encoded']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess features
categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=['float64']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Train model
model = XGBClassifier(objective='multi:softmax', num_class=7, eval_metric='mlogloss')
model.fit(X_train_processed, y_train)

# Evaluate
y_pred = model.predict(X_test_processed)
print(classification_report(y_test, y_pred, target_names=le.classes_))
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")

C:\Users\RQ\AppData\Local\Temp\ipykernel_23792\3086800525.py:49: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


              precision    recall  f1-score   support

  0 - 10 min       0.38      0.84      0.52      3711
 11 - 20 min       0.19      0.02      0.04      1770
 21 - 30 min       0.22      0.09      0.13      1265
 31 - 40 min       0.20      0.03      0.06       985
 41 - 50 min       0.15      0.01      0.01       680
 51 - 60 min       0.11      0.00      0.00       559
     60+ min       0.37      0.34      0.36      2555

    accuracy                           0.36     11525
   macro avg       0.23      0.19      0.16     11525
weighted avg       0.29      0.36      0.27     11525

Accuracy: 0.3649
